# SlowFast fine-tuning

Fine-tunes the `slowfast_r50` model (PyTorchVideo, pretrained on Kinetics-400) on your own action classes.

**What this notebook does**
1. Downloads a small demo dataset (UCF101 subset: 10 classes, ~400 short videos) — or uses your own.
2. Replaces the 400-class Kinetics head with a new head for your classes.
3. Trains, evaluates (accuracy per class + confusion matrix) and saves the weights.

**Runtime:** Colab → *Runtime → Change runtime type → T4 GPU*. With the backbone frozen (default) training takes a few minutes. It also runs on CPU, but much slower.

**Preprocessing is shared with the package:** the notebook imports `src/utils/clip.py` from the SlowFast package repo, so frames are prepared exactly as in the NovaVision executor.

**Your own data:** one folder per action, set `USE_DEMO_DATA = False` and `DATA_ROOT` below (see *Preparing your own videos*):
```
DATA_ROOT/
├── loading_pallet/*.mp4
├── forklift_turning/*.mp4
└── normal/*.mp4
```
The notebook splits the videos into train / validation / test automatically. If you already have your own split, use `DATA_ROOT/train/<class>/`, `DATA_ROOT/val/<class>/`, `DATA_ROOT/test/<class>/` instead.

### Preparing your own videos
- **One action per video**, 2-10 seconds long, the action happening for most of the clip. Cut long recordings into short clips.
- **At least ~20 videos per class**, more is better (50+ gives much more reliable results). Keep the classes roughly balanced.
- **Same conditions as in use:** record with the camera angle, distance and lighting the package will see in production.
- **Add a "normal" class** (nothing special happening) if the package should also tell when none of the actions is present; otherwise the model always picks one of the actions.
- Common formats work: mp4, avi, mov, mkv.

## 1. Setup

In [ ]:
# Colab already has torch, torchvision and opencv. pytorchvideo provides the model definition.
!pip -q install pytorchvideo==0.1.5

import os, sys, glob, random, time, subprocess
import numpy as np
import cv2
import torch
import torch.nn as nn

REPO_URL = "https://github.com/Zerothlaw780/slowfast.git"   # SlowFast package repo (change when it moves)
REPO_BRANCH = "develop"
REPO_DIR = "slowfast_pkg"

if not os.path.isdir(REPO_DIR):
    subprocess.run(["git", "clone", "-q", "-b", REPO_BRANCH, REPO_URL, REPO_DIR], check=True)
sys.path.insert(0, REPO_DIR)

from src.utils.clip import resize_crop, sample_indices, to_inputs, NUM_FRAMES
from src.utils.model import predict

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print("torch", torch.__version__, "| device:", DEVICE)

## 2. Settings

| Name | Meaning |
|---|---|
| `USE_DEMO_DATA`, `DATA_ROOT` | Demo data or the folder with your own videos. On Colab, upload your folder to Google Drive and run the *Google Drive* cell below. |
| `VAL_SPLIT`, `TEST_SPLIT` | Share of each class kept for validation and test when the notebook splits your data. |
| `CLIP_FRAMES` | Consecutive video frames that form one training clip; they are resampled to the 32 frames the model expects. 64 frames ≈ 2.5 s at 25 fps, close to how the model was pretrained. Use the same duration for the package's *Clip Frames* setting. |
| `FREEZE_BACKBONE` | `True`: only the new head is trained (fast, works with little data). `False`: the whole network is trained (slower, needs more data and GPU memory; lower the learning rate). |
| `EPOCHS`, `BATCH_SIZE`, `LR` | Usual training settings. Lower `BATCH_SIZE` if you run out of GPU memory. |

In [ ]:
USE_DEMO_DATA = True            # True: download the UCF101 demo subset; False: use your own videos in DATA_ROOT
DATA_ROOT = "UCF101_subset"     # your data, e.g. "/content/drive/MyDrive/my_actions"
VAL_SPLIT, TEST_SPLIT = 0.15, 0.15   # used only when DATA_ROOT has no train/ val/ test/ folders
CLIP_FRAMES = 64
FREEZE_BACKBONE = True
EPOCHS = 5
BATCH_SIZE = 8
LR = 1e-3
NUM_WORKERS = 2
PRETRAINED = True               # start from Kinetics-400 weights (recommended)
OUTPUT_PATH = "slowfast_finetuned.pth"
SEED = 0

random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)

**Google Drive (Colab only, for your own data):** run this cell to make `/content/drive/MyDrive/...` available. Skip it for the demo data.

In [ ]:
if not USE_DEMO_DATA and DATA_ROOT.startswith("/content/drive"):
    from google.colab import drive
    drive.mount("/content/drive")

## 3. Data
With `USE_DEMO_DATA = True` the UCF101 subset used in the Hugging Face video classification guide (≈150 MB, 10 classes) is downloaded. Your own data is split per class into train / val / test unless it is already split.

In [ ]:
if USE_DEMO_DATA and not os.path.isdir(DATA_ROOT):
    import tarfile
    from huggingface_hub import hf_hub_download
    path = hf_hub_download(repo_id="sayakpaul/ucf101-subset", filename="UCF101_subset.tar.gz", repo_type="dataset")
    with tarfile.open(path) as t:
        t.extractall(".")

VIDEO_EXT = (".mp4", ".avi", ".mov", ".mkv")

assert os.path.isdir(DATA_ROOT), f"DATA_ROOT not found: {DATA_ROOT}"

def videos_by_class(root):
    # {class_name: [video paths]} for every sub-folder of root that contains videos
    out = {}
    for cls in sorted(os.listdir(root)):
        folder = os.path.join(root, cls)
        if os.path.isdir(folder):
            vids = sorted(p for p in glob.glob(os.path.join(folder, "*")) if p.lower().endswith(VIDEO_EXT))
            if vids:
                out[cls] = vids
    return out

if all(os.path.isdir(os.path.join(DATA_ROOT, s)) for s in ("train", "val", "test")):
    by_split = {s: videos_by_class(os.path.join(DATA_ROOT, s)) for s in ("train", "val", "test")}
    CLASSES = sorted(by_split["train"])
    splits = {s: [(p, c) for c, vids in by_split[s].items() for p in vids] for s in by_split}
    print("using the existing train / val / test folders")
else:                                   # one folder per class -> split each class separately
    rng = random.Random(SEED)
    per_class = videos_by_class(DATA_ROOT)
    CLASSES = sorted(per_class)
    splits = {"train": [], "val": [], "test": []}
    for cls, vids in per_class.items():
        vids = vids[:]
        rng.shuffle(vids)
        n_val = max(1, round(len(vids) * VAL_SPLIT))
        n_test = max(1, round(len(vids) * TEST_SPLIT))
        if len(vids) - n_val - n_test < 1:
            raise ValueError(f"class '{cls}' has only {len(vids)} videos; at least 3 are needed (20+ recommended)")
        splits["val"] += [(p, cls) for p in vids[:n_val]]
        splits["test"] += [(p, cls) for p in vids[n_val:n_val + n_test]]
        splits["train"] += [(p, cls) for p in vids[n_val + n_test:]]
    print(f"split each class: {1 - VAL_SPLIT - TEST_SPLIT:.0%} train / {VAL_SPLIT:.0%} val / {TEST_SPLIT:.0%} test")

assert len(CLASSES) >= 2, "at least 2 classes are needed"
CLASS_TO_IDX = {c: i for i, c in enumerate(CLASSES)}
print(len(CLASSES), "classes:", CLASSES)
for s, items in splits.items():
    print(f"{s:5s}: {len(items)} videos")
counts = {c: sum(1 for _, k in splits["train"] if k == c) for c in CLASSES}
small = [c for c, n in counts.items() if n < 15]
if small:
    print("warning: few training videos for", small, "- results will be unreliable")

### Dataset
Each item is one clip: a window of `CLIP_FRAMES` consecutive frames (random position for training, the middle of the video for evaluation), from which 32 evenly spaced frames are taken and prepared with the package's `resize_crop` / `to_inputs`. Training clips are also flipped horizontally at random.

In [ ]:
def read_frames(path):
    cap = cv2.VideoCapture(path)
    frames = []
    while True:
        ok, f = cap.read()
        if not ok:
            break
        frames.append(f)
    cap.release()
    if not frames:
        raise RuntimeError(f"Could not read any frame from {path}")
    return frames

class VideoClips(torch.utils.data.Dataset):
    def __init__(self, items, train):
        self.items, self.train = items, train

    def __len__(self):
        return len(self.items)

    def __getitem__(self, i):
        path, cls = self.items[i]
        frames = read_frames(path)
        n = len(frames)
        window = min(CLIP_FRAMES, n)
        start = random.randint(0, n - window) if self.train else (n - window) // 2
        idx = start + sample_indices(window, NUM_FRAMES)
        clip = [resize_crop(frames[j]) for j in idx]
        if self.train and random.random() < 0.5:
            clip = [np.ascontiguousarray(f[:, ::-1]) for f in clip]
        slow, fast = to_inputs(clip)
        return slow[0], fast[0], CLASS_TO_IDX[cls]

loaders = {
    s: torch.utils.data.DataLoader(VideoClips(items, train=(s == "train")), batch_size=BATCH_SIZE,
                                   shuffle=(s == "train"), num_workers=NUM_WORKERS, pin_memory=(DEVICE == "cuda"))
    for s, items in splits.items()
}
slow, fast, y = next(iter(loaders["train"]))
print("slow", tuple(slow.shape), "fast", tuple(fast.shape), "labels", y.tolist())

## 4. Model
The Kinetics head (`Linear(2304 → 400)`) is replaced with `Linear(2304 → number of classes)`.

In [ ]:
from pytorchvideo.models.hub import slowfast_r50

torch.hub.set_dir("weights")    # the 264 MB Kinetics checkpoint is cached here
model = slowfast_r50(pretrained=PRETRAINED)
head = model.blocks[-1]
head.proj = nn.Linear(head.proj.in_features, len(CLASSES))

if FREEZE_BACKBONE:
    for p in model.parameters():
        p.requires_grad = False
    for p in head.parameters():
        p.requires_grad = True

model = model.to(DEVICE)
trainable = [p for p in model.parameters() if p.requires_grad]
print(f"trainable parameters: {sum(p.numel() for p in trainable):,} / {sum(p.numel() for p in model.parameters()):,}")

def set_train_mode(m):
    m.train()
    if FREEZE_BACKBONE:              # keep BatchNorm statistics of the frozen backbone fixed
        for block in m.blocks[:-1]:
            block.eval()

## 5. Training

In [ ]:
optimizer = torch.optim.AdamW(trainable, lr=LR, weight_decay=1e-4)
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=EPOCHS * len(loaders["train"]))
criterion = nn.CrossEntropyLoss()
use_amp = DEVICE == "cuda"
scaler = torch.amp.GradScaler("cuda", enabled=use_amp)

def evaluate(loader):
    model.eval()
    preds, labels = [], []
    with torch.inference_mode(), torch.autocast(DEVICE, enabled=use_amp):
        for slow, fast, y in loader:
            out = model([slow.to(DEVICE), fast.to(DEVICE)])
            preds += out.argmax(1).cpu().tolist()
            labels += y.tolist()
    preds, labels = np.array(preds), np.array(labels)
    return (preds == labels).mean(), preds, labels

best_acc, history = -1.0, []
for epoch in range(1, EPOCHS + 1):
    set_train_mode(model)
    start, total_loss, seen = time.time(), 0.0, 0
    for slow, fast, y in loaders["train"]:
        slow, fast, y = slow.to(DEVICE), fast.to(DEVICE), y.to(DEVICE)
        with torch.autocast(DEVICE, enabled=use_amp):
            loss = criterion(model([slow, fast]), y)
        optimizer.zero_grad(set_to_none=True)
        scaler.scale(loss).backward()
        scaler.step(optimizer)
        scaler.update()
        scheduler.step()
        total_loss += loss.item() * y.size(0)
        seen += y.size(0)
    val_acc, _, _ = evaluate(loaders["val"])
    history.append((epoch, total_loss / seen, val_acc))
    print(f"epoch {epoch}/{EPOCHS}  loss {total_loss / seen:.3f}  val acc {val_acc:.3f}  ({time.time() - start:.0f} s)")
    if val_acc > best_acc:
        best_acc = val_acc
        torch.save({"state_dict": model.state_dict(), "classes": CLASSES, "arch": "slowfast_r50",
                    "clip_frames": CLIP_FRAMES}, OUTPUT_PATH)
print(f"best val acc {best_acc:.3f} -> saved to {OUTPUT_PATH}")

## 6. Evaluation on the test split
Loads the best checkpoint and reports overall accuracy, accuracy per class and the confusion matrix (rows = true class, columns = predicted class).

In [ ]:
ckpt = torch.load(OUTPUT_PATH, map_location=DEVICE)
model.load_state_dict(ckpt["state_dict"])
test_acc, preds, labels = evaluate(loaders["test"])
print(f"test accuracy: {test_acc:.3f}  ({len(labels)} videos)\n")

cm = np.zeros((len(CLASSES), len(CLASSES)), int)
for t, p in zip(labels, preds):
    cm[t, p] += 1
width = max(len(c) for c in CLASSES)
for i, c in enumerate(CLASSES):
    total = cm[i].sum()
    acc = cm[i, i] / total if total else float("nan")
    print(f"{c:<{width}}  {acc:5.2f}  {cm[i].tolist()}")

## 7. Try it on one video
Same steps the package runs: a clip from the middle of the video → `to_inputs` → `predict` (softmax) → top 3.

In [ ]:
path, true_cls = random.choice(splits["test"])
frames = read_frames(path)
window = min(CLIP_FRAMES, len(frames))
start = (len(frames) - window) // 2
clip = [resize_crop(f) for f in frames[start:start + window]]
probs = predict(model.eval(), to_inputs(clip, DEVICE))
print(os.path.basename(path), "| true:", true_cls)
for r, i in enumerate(np.argsort(probs)[::-1][:3], 1):
    print(f"  {r}. {CLASSES[i]:<20s} {probs[i]:.2f}")

## 8. Using the weights

`slowfast_finetuned.pth` holds `state_dict`, `classes`, `arch` and `clip_frames`. To load it:

```python
ckpt = torch.load("slowfast_finetuned.pth", map_location="cpu")
model = slowfast_r50(pretrained=False)
model.blocks[-1].proj = nn.Linear(2304, len(ckpt["classes"]))
model.load_state_dict(ckpt["state_dict"])
model.eval()
```

The NovaVision package currently uses the Kinetics-400 model. Supporting a custom checkpoint (for example a *Model* config with *Kinetics-400 / Custom* and a weights file) is a planned follow-up.